# 06 — Social Media Charts (Pillow)

Publication-ready PNGs for @unwelcomedata, `twitter_landscape` (1600×900) with watermark, via `shared/chart_factory.py`.

**Initial release — state-level charts:**
1. Top 5 gainers & bottom 5 losers of people (diverging)
2. Biggest one-way moves between states, colored direction labels (ranked)
3. States gaining & losing the most income to migration (diverging)
4. Average income of people each state gains and loses (diverging)
5. Most interstate moves cancel out (diagonal-ceiling scatter)

County-level charts deferred to a phase-2 follow-up (see workspace TODO.md).

Data read **read-only** and passed as DataFrames so this coexists with other open connections.

In [ ]:
import sys, os
from pathlib import Path
import duckdb, yaml

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(PROJECT.parent.parent / 'shared'))

from chart_factory import render_chart
cfg = yaml.safe_load(open('config.yaml'))
conn = duckdb.connect(str(PROJECT / 'data' / 'project.duckdb'), read_only=True)

# Brand colors (see shared/viz.py COOLORS)
TEAL, RED = '#005F73', '#AE2012'
print('✓ factory loaded, DuckDB connected (read-only)')

## 1. Top 5 gainers & bottom 5 losers, 2023 (diverging)
The states with the largest net gain and largest net loss of people. Trimmed from all 51 to the 10 extremes for social legibility.

In [ ]:
# Top 5 net gainers + bottom 5 net losers (full names read better at 10 bars)
net_state = conn.execute("""
    WITH o AS (SELECT origin_state st, SUM(irs_individuals_out) gone FROM migration_flows
               WHERE year=2023 AND origin_type IN ('state','dc') GROUP BY 1),
         i AS (SELECT dest_state st, SUM(irs_individuals_in) came FROM migration_flows
               WHERE year=2023 AND dest_type IN ('state','dc') GROUP BY 1),
         net AS (
             SELECT COALESCE(i.st,o.st) AS st, CAST(COALESCE(came,0)-COALESCE(gone,0) AS BIGINT) AS net
             FROM i FULL OUTER JOIN o ON i.st=o.st
         )
    SELECT sr.name AS state, n.net
    FROM net n JOIN state_ref sr ON sr.abbrev = n.st
    WHERE n.st IN (
        (SELECT st FROM net ORDER BY net DESC LIMIT 5)
        UNION ALL
        (SELECT st FROM net ORDER BY net ASC LIMIT 5)
    )
    ORDER BY n.net DESC
""").df()
net_state['label'] = net_state['net'].map(lambda v: f'{"+" if v>=0 else "−"}{abs(v):,.0f}')

render_chart({
    'type': 'diverging_bars', 'cfg': cfg, 'preset': 'twitter_landscape',
    'table': net_state, 'category_col': 'state', 'value_col': 'net', 'label_col': 'label',
    'title': 'The 5 states gaining the most people — and the 5 losing the most',
    'subtitle': 'Net migration in 2023: arrivals minus departures (IRS tax-return data)',
    'source': 'IRS SOI migration 2022–2023 · net = inflow-file arrivals minus outflow-file departures',
    'pos_color': TEAL, 'neg_color': RED,
    'filename': '01_net_migration_by_state',
})

## 2. Biggest one-way moves between state pairs, 2023 (ranked)
Net flow per state pair, collapsed to a single winning direction. Red state lost people to the blue state.

In [ ]:
pairs = conn.execute("""
    WITH s AS (SELECT origin_state a, dest_state b, irs_individuals_out n FROM migration_flows
               WHERE year=2023 AND origin_type IN ('state','dc') AND dest_type IN ('state','dc') AND irs_individuals_out IS NOT NULL),
    paired AS (SELECT f.a,f.b,f.n a_to_b, COALESCE(r.n,0) b_to_a FROM s f LEFT JOIN s r ON r.a=f.b AND r.b=f.a WHERE f.a<f.b)
    SELECT CASE WHEN a_to_b>=b_to_a THEN a ELSE b END AS loser,
           CASE WHEN a_to_b>=b_to_a THEN b ELSE a END AS winner,
           CAST(ABS(a_to_b-b_to_a) AS BIGINT) AS net_flow
    FROM paired ORDER BY net_flow DESC LIMIT 12
""").df()
pairs['direction'] = pairs['loser'] + '  →  ' + pairs['winner']   # plain fallback label
pairs['label'] = pairs['net_flow'].map(lambda v: f'+{v:,.0f}')
# Colored label segments: red loser → (gray arrow) → teal winner
pairs['segments'] = pairs.apply(lambda r: [(r['loser'], RED), ('  →  ', '#9CA3AF'), (r['winner'], TEAL)], axis=1)

render_chart({
    'type': 'single_ranked_bars', 'cfg': cfg, 'preset': 'twitter_landscape',
    'table': pairs, 'category_col': 'direction', 'value_col': 'net_flow', 'label_col': 'label',
    'label_segments_col': 'segments',
    'title': 'The biggest one-way moves between states, 2023',
    'subtitle': 'Net migration for each state pair — red state lost people to the blue state',
    'source': 'IRS SOI migration 2022–2023',
    'bar_color': TEAL,
    'filename': '02_net_directed_pairs',
})

## 3. The states gaining & losing the most income to migration (diverging)
Net AGI: the tax-base view. Top 5 and bottom 5 states. Florida's gain dwarfs the field.

In [ ]:
# Top 5 AGI gainers + bottom 5 AGI losers (full names, 10 bars for legibility)
agi = conn.execute("""
    WITH o AS (SELECT origin_state st, SUM(irs_agi_out) a FROM migration_flows WHERE year=2023 AND origin_type IN ('state','dc') GROUP BY 1),
         i AS (SELECT dest_state st, SUM(irs_agi_in) a FROM migration_flows WHERE year=2023 AND dest_type IN ('state','dc') GROUP BY 1),
         net AS (SELECT COALESCE(i.st,o.st) AS st, (COALESCE(i.a,0)-COALESCE(o.a,0))/1e6 AS net_agi_billion
                 FROM i FULL OUTER JOIN o ON i.st=o.st)
    SELECT sr.name AS state, n.net_agi_billion
    FROM net n JOIN state_ref sr ON sr.abbrev = n.st
    WHERE n.st IN (
        (SELECT st FROM net ORDER BY net_agi_billion DESC LIMIT 5)
        UNION ALL
        (SELECT st FROM net ORDER BY net_agi_billion ASC LIMIT 5)
    )
    ORDER BY n.net_agi_billion DESC
""").df()
agi['label'] = agi['net_agi_billion'].map(lambda v: f'{"+" if v>=0 else "−"}${abs(v):.1f}B')

render_chart({
    'type': 'diverging_bars', 'cfg': cfg, 'preset': 'twitter_landscape', 'table': agi,
    'category_col': 'state', 'value_col': 'net_agi_billion', 'label_col': 'label',
    'title': 'The states gaining — and losing — the most income to migration',
    'subtitle': 'Net adjusted gross income moved in or out with migrants, 2023 (billions of dollars)',
    'source': 'IRS SOI migration 2022–2023 · AGI in origin-year dollars · net = inflow-file minus outflow-file',
    'pos_color': TEAL, 'neg_color': RED,
    'filename': '03_net_agi_by_state',
})

## 4. The average income of the people each state gains and loses (diverging)
AGI per net migrant, 2023 — right: income arriving per net newcomer · left: income leaving per net departure. States with net |migration| ≥ 20,000.

In [ ]:
import numpy as np
perp = conn.execute("""
    WITH op AS (SELECT origin_state st, SUM(irs_individuals_out) p, SUM(irs_agi_out) a FROM migration_flows WHERE year=2023 AND origin_type IN ('state','dc') GROUP BY 1),
         ip AS (SELECT dest_state st, SUM(irs_individuals_in) p, SUM(irs_agi_in) a FROM migration_flows WHERE year=2023 AND dest_type IN ('state','dc') GROUP BY 1)
    SELECT COALESCE(ip.st,op.st) state, COALESCE(ip.p,0)-COALESCE(op.p,0) net_people,
           (COALESCE(ip.a,0)-COALESCE(op.a,0))*1000.0 net_agi
    FROM ip FULL OUTER JOIN op ON ip.st=op.st
""").df()
perp = perp[perp.net_people.abs() >= 20000].copy()
perp['per_k'] = ((perp.net_agi.abs()/perp.net_people.abs()) * np.sign(perp.net_people)) / 1000
perp['label'] = perp['per_k'].map(lambda v: f'{"+" if v>=0 else "−"}${abs(v):.0f}k')

render_chart({
    'type': 'diverging_bars', 'cfg': cfg, 'preset': 'twitter_landscape', 'table': perp,
    'category_col': 'state', 'value_col': 'per_k', 'label_col': 'label',
    'title': 'The average income of people each state gains and loses',
    'subtitle': 'AGI per net migrant, 2023 — right: income arriving per net newcomer · left: income leaving per net departure',
    'source': 'IRS SOI migration 2022–2023 · states with net |migration| ≥ 20,000',
    'pos_color': TEAL, 'neg_color': RED,
    'filename': '04_agi_per_migrant',
})

## 5. Most interstate moves cancel out (diagonal-ceiling scatter)
The impossible area above the diagonal is removed; net is read off horizontal lines labeled where they meet the ceiling. Red state lost people to the blue state.

In [ ]:
nv_c = conn.execute("""
    WITH s AS (SELECT origin_state a, dest_state b, irs_individuals_out n FROM migration_flows
               WHERE year=2023 AND origin_type IN ('state','dc') AND dest_type IN ('state','dc') AND irs_individuals_out IS NOT NULL),
    paired AS (SELECT f.a,f.b,f.n a_to_b, COALESCE(r.n,0) b_to_a FROM s f LEFT JOIN s r ON r.a=f.b AND r.b=f.a WHERE f.a<f.b)
    SELECT CASE WHEN a_to_b>=b_to_a THEN a ELSE b END AS loser,
           CASE WHEN a_to_b>=b_to_a THEN b ELSE a END AS winner,
           (a_to_b+b_to_a) AS gross_flow, ABS(a_to_b-b_to_a) AS net_flow
    FROM paired
""").df()
# Colored callout: red loser → teal winner (matches chart 2)
nv_c['segments'] = nv_c.apply(lambda r: [(r['loser'], RED), (' → ', '#9CA3AF'), (r['winner'], TEAL)], axis=1)

render_chart({
    'type': 'scatter_ceiling', 'cfg': cfg, 'preset': 'twitter_landscape', 'table': nv_c,
    'x_col': 'gross_flow', 'y_col': 'net_flow', 'label_segments_col': 'segments', 'label_top_n': 6, 'n_ref_lines': 5,
    'title': 'Most interstate moves cancel each other out',
    'subtitle': 'Each dot is a state pair. A pair can never have more net migration than total migration — the diagonal is a ceiling almost no one reaches.',
    'source': 'IRS SOI migration 2022–2023',
    'point_color': '#0A9396', 'point_alpha': 170, 'ceiling_color': '#003049', 'ceiling_label': 'ceiling: 100% one-way',
    'ref_fmt': (lambda v: f'{v/1e3:.0f}k net'),
    'filename': '05_net_vs_gross',
})

---
## Summary & Cleanup

In [ ]:
conn.close()
social_dir = PROJECT / 'outputs' / 'social'
pngs = sorted(social_dir.glob('*.png'))
print(f'✓ {len(pngs)} charts in {social_dir}:')
for png in pngs:
    print(f'  • {png.name} ({png.stat().st_size/1024:.0f} KB)')